In [15]:
import os
import time
import pandas as pd

# Java 11 via sdkman
os.environ['JAVA_HOME'] = '/usr/local/sdkman/candidates/java/current'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor, RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder \
    .appName("ModeleImmo") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)

Spark OK : 3.5.0


In [16]:
bases_fusionnees = spark.read.csv("bases_fusionnees.csv", header=True, inferSchema=True, sep=",")

In [17]:
bases_fusionnees.columns

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']

In [18]:

bases_fusionnees.describe().show()
bases_fusionnees.groupBy("type_bien").count().orderBy(F.desc("count")).show()
bases_fusionnees.groupBy("pieces").count().orderBy("pieces").show()
bases_fusionnees.groupBy("ville").count().orderBy(F.desc("count")).show(10)

+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|summary|  type_bien|            pieces|             prix|           surface|             etage| ville|       code_postal|
+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|  count|       8019|              8019|             8019|              8019|              8019|  8019|              8019|
|   mean|       NULL|  4.05100386581868|427315.1996508293|101.58756702830776|2.8797779201366644|  NULL|  69273.8709315376|
| stddev|       NULL|1.6509496507084573|300296.6495465243| 61.13174260486775|  2.33179621631308|  NULL|252.03671516019935|
|    min|appartement|               1.0|          49000.0|               9.0|                -1|affoux|             69001|
|    max|     maison|              12.0|        4160000.0|             900.0|               RDC| éveux|             69970|
+-------+-------

In [19]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)
train_data = train_data.cache()
test_data  = test_data.cache()
print(f"Taille train : {train_data.count()} lignes")
print(f"Taille test  : {test_data.count()} lignes")

Taille train : 6486 lignes
Taille test  : 1533 lignes


In [20]:

indexer_ville = StringIndexer(inputCol="ville",       outputCol="ville_index",  handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_clean",  outputCol="type_index",   handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage_clean", outputCol="etage_index",  handleInvalid="keep")

features_cols = [
    "surface", "pieces", "chambres", "terrain", "neuf_flag",
    "ville_index", "type_index", "etage_index"
]

assembler = VectorAssembler(inputCols=features_cols, outputCol="features")

print("Features utilisees :")
for f in features_cols:
    print(f"  - {f}")

Features utilisees :
  - surface
  - pieces
  - chambres
  - terrain
  - neuf_flag
  - ville_index
  - type_index
  - etage_index


In [21]:
bases_fusionnees.printSchema()
print(bases_fusionnees.columns)

root
 |-- type_bien: string (nullable = true)
 |-- pieces: double (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- etage: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: integer (nullable = true)

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']


In [22]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)

In [23]:
train_data

DataFrame[type_bien: string, pieces: double, prix: double, surface: double, etage: string, ville: string, code_postal: int]

In [24]:
#Encodage variables categorilles
indexer_ville = StringIndexer(inputCol="ville", outputCol="ville_idx", handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_bien", outputCol="type_idx", handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage", outputCol="etage_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["type_idx", "pieces", "surface", "etage_idx", "ville_idx"],
    outputCol="features"
)

In [25]:
# Arbre de decision
dt = DecisionTreeRegressor(featuresCol="features", labelCol="prix", maxDepth=8, maxBins=256, seed=42)
pipeline_dt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, dt])
model_dt = pipeline_dt.fit(train_data)
print("Decision Tree entraine")

Decision Tree entraine


In [26]:
#  Random Forest
rf = RandomForestRegressor(
    featuresCol="features", labelCol="prix",
    numTrees=100, maxDepth=10, maxBins=256, seed=42
)
pipeline_rf = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, rf])
model_rf = pipeline_rf.fit(train_data)
print("Random Forest entraine")

Random Forest entraine


In [27]:
# GBT (Gradient Boosted Trees)
gbt = GBTRegressor(
    featuresCol="features", labelCol="prix",
    maxIter=100, maxDepth=8, maxBins=256, stepSize=0.1, seed=42
)
pipeline_gbt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, gbt])
model_gbt = pipeline_gbt.fit(train_data)
print("GBT entraine")

GBT entraine


In [28]:

dt_pred  = model_dt.transform(test_data)
rf_pred  = model_rf.transform(test_data)
gbt_pred = model_gbt.transform(test_data)

# Evaluation
r2_eval   = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="mae")

#  métriques
dt_r2,  dt_rmse,  dt_mae  = r2_eval.evaluate(dt_pred),  rmse_eval.evaluate(dt_pred),  mae_eval.evaluate(dt_pred)
rf_r2,  rf_rmse,  rf_mae  = r2_eval.evaluate(rf_pred),  rmse_eval.evaluate(rf_pred),  mae_eval.evaluate(rf_pred)
gbt_r2, gbt_rmse, gbt_mae = r2_eval.evaluate(gbt_pred), rmse_eval.evaluate(gbt_pred), mae_eval.evaluate(gbt_pred)


resultats = pd.DataFrame({
    "Modele":        ["Decision Tree", "Random Forest", "GBT"],
    "R2":            [round(dt_r2, 4),   round(rf_r2, 4),   round(gbt_r2, 4)],
    "RMSE (euros)":  [round(dt_rmse),    round(rf_rmse),    round(gbt_rmse)],
    "MAE (euros)":   [round(dt_mae),     round(rf_mae),     round(gbt_mae)]
})

print(resultats.to_string(index=False))

# Meilleur modele
print(f"Le meilleur modele est : {meilleur}")
print(f"R2 le plus eleve   : {round(rf_r2, 4)} il explique {round(rf_r2*100, 1)}% de la variance des prix")
print(f"RMSE la plus faible : {round(rf_rmse):,} c'est l'erreur moyenne la plus petite")

       Modele     R2  RMSE (euros)  MAE (euros)
Decision Tree 0.5761        200175       105386
Random Forest 0.6408        184267       106517
          GBT 0.5112        214940       113177
Le meilleur modele est : Random Forest
R2 le plus eleve   : 0.6408 il explique 64.1% de la variance des prix
RMSE la plus faible : 184,267 c'est l'erreur moyenne la plus petite
